# SNNAI v6.7.0 ablation: 5 patterns x seeds (light/full)

Measures SNN val_ppl across feature ablations with **fair** parameter
matching (TransformerBaseline auto-sized to the SNN budget) and optional
knowledge distillation. The `LIGHT` flag at the top switches between a
cheap gate run and the full 3-seed run.

Key fix vs v6.6.2: SpikingSelfAttention._normalize backward no longer
produces a 0*inf NaN (now `(var+eps).sqrt()`), so the all-features SNN
trains instead of going to loss=nan / dead spikes.


In [ ]:
# Install and clone the v6.7.0 tag.
# The kernel is pushed with acc='NvidiaTeslaT4' so Kaggle assigns a T4
# (compute capability 7.5), which the default PyTorch build supports
# directly. No torch reinstall / capability detection needed.
!pip install -q numpy snntorch
!rm -rf snnai
!git clone --depth 1 --branch v6.7.0 https://github.com/sabunosuke1008-create/snnai.git
import sys
sys.path.insert(0, 'snnai')


In [ ]:
import os, json, time, math, requests
import torch
from snnai.utils.download_corpus import download_wikitext2
from snnai.modules.language.bpe_tokenizer import BPETokenizer
from snnai.modules.language.large_scale_lm import LargeScaleSNNLM
from snnai.benchmarks.transformer_comparison import (
    TransformerBaseline, fair_compare, build_matched_transformer)
from snnai.benchmarks.large_corpus_trainer import CharLMDataset, collate_fn
from torch.utils.data import DataLoader

# ---- switch: True = cheap gate, False = full 3-seed run ----
LIGHT = False

if LIGHT:
    EMBED, HID, NLAY = 64, 64, 1
    SEQ, BATCH, TSTEPS, EPOCHS = 48, 16, 6, 2
    VOCAB = 512
    SEEDS = (0,)
    USE_DISTILL = False
    USE_WIKITEXT = False
else:
    EMBED, HID, NLAY = 128, 512, 3
    SEQ, BATCH, TSTEPS, EPOCHS = 128, 32, 20, 5
    VOCAB = 2048
    SEEDS = (0, 1, 2)
    USE_DISTILL = True
    USE_WIKITEXT = True

# Generation feeds up to 128 tokens as one sequence, so keep this >= 128.
MAX_SEQ = 256
# Pushed on a T4 (acc='NvidiaTeslaT4'), so CUDA is available and supported.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', device, 'LIGHT', LIGHT, 'SEEDS', SEEDS)


In [ ]:
t0 = time.time()
corpus = requests.get(
    'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt').text
if USE_WIKITEXT:
    wt_root = download_wikitext2(dest_dir='/kaggle/working/wikitext-2', timeout=180)
    if wt_root is not None:
        wt_text = ''.join(
            open(os.path.join(wt_root, f)).read()
            for f in ['wiki.train.raw', 'wiki.valid.raw', 'wiki.test.raw']
            if os.path.exists(os.path.join(wt_root, f)))
        if wt_text:
            corpus = corpus + '\n' + wt_text
print('corpus length', len(corpus))
bpe = BPETokenizer([corpus], vocab_size=VOCAB, max_train_bytes=2_000_000)
print('bpe vocab', bpe.vocab_size)

def probe_firing_rate(model, text, tokenizer, seq_len, batch_size, time_steps, n_batches=4):
    ds = CharLMDataset(text, tokenizer, seq_len=seq_len)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True,
                        collate_fn=lambda b: collate_fn(b, tokenizer.vocab_size))
    model.eval()
    rates = []
    with torch.no_grad():
        for i, (one_hot, targets) in enumerate(loader):
            if i >= n_batches:
                break
            x = one_hot.unsqueeze(0).repeat(time_steps, 1, 1, 1).to(device)
            if hasattr(model, 'reset_memory'):
                model.reset_memory()
            out, spikes = model(x, return_spikes=True)
            layer_rates = [s.float().mean().item() for s in spikes]
            rates.append(sum(layer_rates) / max(1, len(layer_rates)))
    return sum(rates) / max(1, len(rates))


In [ ]:
patterns = [
    ('P1_baseline',     dict(use_sequence_recurrent=True,  use_positional_encoding=False, use_hippocampus_gate=False, use_spiking_attention=False)),
    ('P2_posenc',       dict(use_sequence_recurrent=True,  use_positional_encoding=True,  use_hippocampus_gate=False, use_spiking_attention=False)),
    ('P3_hippocampus',  dict(use_sequence_recurrent=True,  use_positional_encoding=True,  use_hippocampus_gate=True,  use_spiking_attention=False)),
    ('P4_attention',    dict(use_sequence_recurrent=True,  use_positional_encoding=True,  use_hippocampus_gate=False, use_spiking_attention=True)),
    ('P5_all_features', dict(use_sequence_recurrent=True,  use_positional_encoding=True,  use_hippocampus_gate=True,  use_spiking_attention=True)),
]

def build_snn(flags):
    return LargeScaleSNNLM(
        bpe.vocab_size, embed_dim=EMBED, hidden_dim=HID, num_layers=NLAY,
        output_mode='mem_last', max_seq_len=MAX_SEQ,
        ssa_input='spike', enable_ssa_residual=True, enable_ssa_layernorm=True,
        **flags)

results = {}
for name, flags in patterns:
    torch.manual_seed(0)
    snn = build_snn(flags)
    snn_p = sum(p.numel() for p in snn.parameters())
    tf = build_matched_transformer(snn, bpe.vocab_size)
    t_pattern = time.time()
    res = fair_compare(
        corpus, bpe, snn, tf, epochs=EPOCHS, seq_len=SEQ, batch_size=BATCH,
        time_steps=TSTEPS, device=device, seeds=SEEDS,
        match_transformer=True,
        use_distill=(USE_DISTILL and name == 'P5_all_features'),
        distill_epochs=EPOCHS, save_dir='/kaggle/working', verbose=True)
    fr = probe_firing_rate(snn, corpus, bpe, SEQ, BATCH, TSTEPS)
    snn_vp = res['snn_history']['val_ppl']
    mean = res.get('snn_val_ppl_mean')
    std = res.get('snn_val_ppl_std')
    results[name] = {
        'flags': flags,
        'snn_params': snn_p,
        'tf_params': res['transformer_parameters'],
        'snn_val_ppl': snn_vp,
        'snn_val_ppl_mean': mean,
        'snn_val_ppl_std': std,
        'tf_val_ppl': res['transformer_history']['val_ppl'],
        'snn_firing_rate': fr,
        'elapsed_sec': round(time.time() - t_pattern, 1),
    }
    print('===', name, '===', json.dumps(results[name], indent=2, default=str))


In [ ]:
print('\n===== ABLATION SUMMARY =====')
for name, r in results.items():
    mean = r['snn_val_ppl_mean'] if r['snn_val_ppl_mean'] is not None else r['snn_val_ppl'][-1]
    std = r['snn_val_ppl_std'] if r['snn_val_ppl_std'] is not None else 0.0
    print('%s snn_val_ppl=%.2f+-%.2f  tf_val_ppl=%.2f  firing=%.4f  snn_params=%d  tf_params=%d'
          % (name.ljust(16), mean, std, r['tf_val_ppl'][-1], r['snn_firing_rate'],
             r['snn_params'], r['tf_params']))
with open('/kaggle/working/ablation_summary.json', 'w') as f:
    json.dump({'light': LIGHT, 'seeds': list(SEEDS), 'results': results}, f, indent=2, default=str)
print('elapsed_total_sec', round(time.time() - t0, 1))
